## Diversity Analysis


In [2]:
!git clone https://github.com/DimitrisKu/Active-Reading--Pattern-Recognition.git

import os

# %cd /kaggle/working/Active-Reading--Pattern-Recognition

%cd /content/Active-Reading--Pattern-Recognition

os.getcwd()

fatal: destination path 'Active-Reading--Pattern-Recognition' already exists and is not an empty directory.
/content/Active-Reading--Pattern-Recognition


'/content/Active-Reading--Pattern-Recognition'

In [21]:
from pathlib import Path
import json, re

BASE = Path("Finetune_Datasets/simplewiki")

datasets = {
    "active_reading": "active_reading_dataset.jsonl",
    "qa": "qa_dataset.jsonl",
    "paraphrase": "paraphrase_dataset.jsonl",
}


[active_reading] rows=1593 unique=735 unique_norm=735
  -> wrote: Finetune_Datasets/simplewiki/docname_exports/active_reading_doc_names_unique.txt
  -> wrote: Finetune_Datasets/simplewiki/docname_exports/active_reading_doc_names_unique_normalized.txt
  -> wrote: Finetune_Datasets/simplewiki/docname_exports/active_reading_doc_names_all_raw.txt
[qa] rows=47832 unique=855 unique_norm=855
  -> wrote: Finetune_Datasets/simplewiki/docname_exports/qa_doc_names_unique.txt
  -> wrote: Finetune_Datasets/simplewiki/docname_exports/qa_doc_names_unique_normalized.txt
  -> wrote: Finetune_Datasets/simplewiki/docname_exports/qa_doc_names_all_raw.txt
[paraphrase] rows=1202 unique=856 unique_norm=856
  -> wrote: Finetune_Datasets/simplewiki/docname_exports/paraphrase_doc_names_unique.txt
  -> wrote: Finetune_Datasets/simplewiki/docname_exports/paraphrase_doc_names_unique_normalized.txt
  -> wrote: Finetune_Datasets/simplewiki/docname_exports/paraphrase_doc_names_all_raw.txt


# Active Reading, Paraphrase, QA

In [59]:
import json, re
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np
import pandas as pd

BASE = Path("Finetune_Datasets/simplewiki")

DATASETS = {
    "active_reading": BASE / "active_reading_dataset.jsonl",
    "qa":            BASE / "qa_dataset.jsonl",
    "paraphrase":    BASE / "paraphrase_dataset.jsonl",
}

def canon_doc_name(doc: str, method: str) -> str:
    doc = "" if doc is None else str(doc)
    if method == "qa":
        doc = doc.replace("_", " ")
    doc = doc.strip()
    doc = re.sub(r"\s+", " ", doc)
    return doc

def get_text(rec, method):
    if method == "qa":
        q = (rec.get("question") or "").strip()
        a = (rec.get("answer") or "").strip()
        return f"Q: {q}\nA: {a}".strip()
    if method == "paraphrase":
        return (rec.get("text") or "").strip()
    return (rec.get("active_reading") or "").strip()

def summarize(arr):
    arr = np.array(arr, dtype=np.int64)
    if arr.size == 0:
        return {}
    return {
        "n": int(arr.size),
        "min": int(arr.min()),
        "p10": int(np.percentile(arr, 10)),
        "p25": int(np.percentile(arr, 25)),
        "p50": int(np.percentile(arr, 50)),
        "p75": int(np.percentile(arr, 75)),
        "p90": int(np.percentile(arr, 90)),
        "max": int(arr.max()),
        "mean": float(arr.mean()),
    }

# bins: όπως πριν (μη αθροιστικά buckets via first-fit)
BINS = [1,2,3,4,5,10,20,50,100,200,500,1000]

reports = []
hist_rows = []
ge_k_rows = []
collision_rows = []

for method, path in DATASETS.items():
    doc_counts = Counter()          # canonical doc -> count
    raw_doc_counts = Counter()      # raw doc -> count
    canon_to_raw = defaultdict(set) # canonical doc -> set(raw docs)
    lengths = []
    empty_text = 0
    missing_doc = 0

    with open(path, "r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            rec = json.loads(line)

            raw_doc = rec.get("doc_name", None)
            if raw_doc is None:
                missing_doc += 1
                continue

            raw_doc = str(raw_doc)
            doc = canon_doc_name(raw_doc, method)

            raw_doc_counts[raw_doc] += 1
            doc_counts[doc] += 1
            canon_to_raw[doc].add(raw_doc)

            text = get_text(rec, method)
            if text:
                lengths.append(len(text.split()))
            else:
                empty_text += 1

    counts = list(doc_counts.values())

    # collisions after canonicalization (only matters for QA typically)
    collisions = sum(1 for canon, raws in canon_to_raw.items() if len(raws) > 1)
    collision_rows.append({
        "method": method,
        "raw_unique_docs": int(len(raw_doc_counts)),
        "canonical_unique_docs": int(len(doc_counts)),
        "canonicalization_collisions": int(collisions),
    })

    # docs with >=k outputs
    for k in [2,3,4,5,6]:
        ge = int(sum(1 for c in counts if c >= k))
        ge_k_rows.append({"method": method, "k": k, "docs_ge_k": ge})

    # method summary
    reports.append({
        "method": method,
        "total_rows": int(sum(counts)),
        "unique_docs_canonical": int(len(doc_counts)),
        "empty_text_rows": int(empty_text),
        "missing_doc_name_rows": int(missing_doc),
        "outputs_per_doc_min": int(np.min(counts)) if counts else 0,
        "outputs_per_doc_p50": int(np.percentile(counts, 50)) if counts else 0,
        "outputs_per_doc_p90": int(np.percentile(counts, 90)) if counts else 0,
        "outputs_per_doc_max": int(np.max(counts)) if counts else 0,
        **{f"len_words_{kk}": vv for kk, vv in summarize(lengths).items()},
    })

    # histogram outputs/doc
    bin_counts = {f"<= {b}": 0 for b in BINS}
    bin_counts["> 1000"] = 0
    for c in counts:
        placed = False
        for b in BINS:
            if c <= b:
                bin_counts[f"<= {b}"] += 1
                placed = True
                break
        if not placed:
            bin_counts["> 1000"] += 1

    for bin_label, v in bin_counts.items():
        hist_rows.append({"method": method, "bin": bin_label, "docs": int(v)})

rep_df = pd.DataFrame(reports).sort_values("method").reset_index(drop=True)
hist_df = pd.DataFrame(hist_rows)
ge_k_df = pd.DataFrame(ge_k_rows).sort_values(["method","k"]).reset_index(drop=True)
canon_df = pd.DataFrame(collision_rows).sort_values("method").reset_index(drop=True)

print("=== SANITY SUMMARY (per method) ===")
display(rep_df)

print("\n=== DOC_NAME CANONICALIZATION CHECK ===")
display(canon_df)

print("\n=== DOCS WITH >= k OUTPUTS ===")
display(ge_k_df)

print("\n=== OUTPUTS/DOC HISTOGRAM (pivot) ===")
pivot = hist_df.pivot_table(index="bin", columns="method", values="docs", aggfunc="sum", fill_value=0)
display(pivot)


=== SANITY SUMMARY (per method) ===


,method,total_rows,unique_docs_canonical,empty_text_rows,missing_doc_name_rows,outputs_per_doc_min,outputs_per_doc_p50,outputs_per_doc_p90,outputs_per_doc_max,len_words_n,len_words_min,len_words_p10,len_words_p25,len_words_p50,len_words_p75,len_words_p90,len_words_max,len_words_mean
0,active_reading,1593,735,0,0,1,2,4,13,1593,31,435,548,611,662,701,789,586.318895
1,paraphrase,1202,856,0,0,1,1,3,9,1202,9,126,290,667,1303,1872,2956,850.663062
2,qa,47832,855,0,0,1,29,137,771,47832,6,13,16,21,27,36,220,23.213853



=== DOC_NAME CANONICALIZATION CHECK ===


,method,raw_unique_docs,canonical_unique_docs,canonicalization_collisions
0,active_reading,735,735,0
1,paraphrase,856,856,0
2,qa,855,855,0



=== DOCS WITH >= k OUTPUTS ===


,method,k,docs_ge_k
0,active_reading,2,431
1,active_reading,3,187
2,active_reading,4,86
3,active_reading,5,59
4,active_reading,6,39
5,paraphrase,2,173
6,paraphrase,3,93
7,paraphrase,4,45
8,paraphrase,5,19
9,paraphrase,6,10



=== OUTPUTS/DOC HISTOGRAM (pivot) ===


method,active_reading,paraphrase,qa
bin,,,
<= 1,304,683,9
<= 10,37,10,94
<= 100,0,0,147
<= 1000,0,0,3
<= 2,244,80,8
<= 20,2,0,176
<= 200,0,0,83
<= 3,101,48,8
<= 4,27,26,17


Για την αξιολόγηση του diversity ανά doc, απαιτείται αντιστοίχιση των συνόλων ώστε οι συγκρίσεις μεταξύ μεθόδων να μην επηρεάζονται από συγχυτικούς παράγοντες όπως το πλήθος και το μήκος των παραγόμενων εξόδων. Παρατηρείται ότι οι μέθοδοι διαφέρουν τόσο στον αριθμό εξόδων ανά doc_name (ιδίως το QA) όσο και στο μήκος των εξόδων (QA τάξης 10, paraphrase - active reading ταξης 100). Δεδομένου ότι η μετρικές  Self-BLEU και n-gram–based είναι ευαίσθητες στο μήκος, υιοθετούμε αξιολόγηση υπό σταθερό προϋπολογισμό (fixed-budget): για κάθε μέθοδο επιλέγουμε κοινό υποσύνολο εγγράφων που διαθέτουν τουλάχιστον
k εξόδους, δειγματοληπτούμε ακριβώς
k εξόδους ανά έγγραφο και περικόπτουμε κάθε έξοδο σε
T λέξεις.

 Με τον τρόπο αυτό, αναλύουμε εντός του ίδιου doc_name, σε ισοδύναμες συνθήκες, ακολουθώντας τη λογική χρησιμοποιείται στη Paper. Για τη μέθοδο QA, όπου υπάρχει μεγάλος αριθμός διαθέσιμων εξόδων, εφαρμόζουμε επαναληπτική υποδειγματοληψία (repeated subsampling) ώστε να εκτιμήσουμε τη διακύμανση που οφείλεται στην επιλογή των k εξόδων και να αναφέρουμε στατιστικά σταθερά αποτελέσματα.

In [66]:
import json, re, random
from pathlib import Path
from collections import defaultdict
import numpy as np

BASE = Path("Finetune_Datasets/simplewiki")

DATASETS = {
    "active_reading": BASE / "active_reading_dataset.jsonl",
    "qa":            BASE / "qa_dataset.jsonl",
    "paraphrase":    BASE / "paraphrase_dataset.jsonl",
}

OUTDIR = BASE / "eval_sets"
OUTDIR.mkdir(parents=True, exist_ok=True)

# ====== PARAMS ======
K = 3
M = 78
T_LIST = [128, 256]      # budgets (words per output)
SEED = 42
QA_REPEATS = 50          # how many alternative QA subsets to export (for uncertainty)
DO_CLEAN = True          # light cleaning
# ====================

def canon_doc_name(doc: str, method: str) -> str:
    doc = "" if doc is None else str(doc)
    if method == "qa":
        doc = doc.replace("_", " ")
    doc = doc.strip()
    doc = re.sub(r"\s+", " ", doc)
    return doc

def get_text(rec, method):
    if method == "qa":
        q = (rec.get("question") or "").strip()
        a = (rec.get("answer") or "").strip()
        return f"Q: {q}\nA: {a}".strip()
    if method == "paraphrase":
        return (rec.get("text") or "").strip()
    return (rec.get("active_reading") or "").strip()

def clean_basic(text: str) -> str:
    # light, method-agnostic cleanup (keep identical for all)
    text = re.sub(r"^#+\s*", "", text, flags=re.MULTILINE)        # markdown headers
    text = re.sub(r"^---+\s*$", " ", text, flags=re.MULTILINE)    # horizontal rules
    text = re.sub(r"\s+", " ", text).strip()
    return text

def truncate_random_window(text: str, T: int, seed_key):
    w = text.split()
    if len(w) <= T:
        return text
    rng = random.Random(hash(seed_key))
    start = rng.randrange(0, len(w) - T + 1)
    return " ".join(w[start:start+T])

def load_grouped(method: str, path: Path):
    """
    Returns:
      grouped[canon_doc] = list of raw outputs (strings)
    """
    grouped = defaultdict(list)
    with open(path, "r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            rec = json.loads(line)
            raw_doc = rec.get("doc_name", None)
            if raw_doc is None:
                continue
            doc = canon_doc_name(raw_doc, method)
            text = get_text(rec, method)
            if not text:
                continue
            grouped[doc].append(text)
    return grouped

def pick_k(texts, k, seed_key):
    rng = random.Random(seed_key)
    idx = list(range(len(texts)))
    rng.shuffle(idx)
    picks = [texts[i] for i in idx[:k]]
    return picks

# ---- Load all ----
G = {}
for method, path in DATASETS.items():
    G[method] = load_grouped(method, path)
    print(f"{method}: docs={len(G[method])}")

# ---- Find eligible docs: present in all + >=K outputs in all ----
docs_all = set(G["active_reading"].keys()) & set(G["qa"].keys()) & set(G["paraphrase"].keys())
eligible = []
for d in docs_all:
    if len(G["active_reading"][d]) >= K and len(G["qa"][d]) >= K and len(G["paraphrase"][d]) >= K:
        eligible.append(d)

eligible = sorted(eligible)
print("docs in all 3:", len(docs_all))
print(f"eligible (>=K={K} in all 3):", len(eligible))

if len(eligible) < M:
    raise ValueError(f"Not enough eligible docs for M={M}. Have {len(eligible)}. Reduce M or K.")

# ---- Choose M docs deterministically ----
rng = random.Random(SEED)
chosen_docs = eligible.copy()
rng.shuffle(chosen_docs)
chosen_docs = sorted(chosen_docs[:M])
print("chosen_docs:", len(chosen_docs))

# Save doc list for reproducibility
meta_path = OUTDIR / f"meta_k{K}_M{M}_seed{SEED}.json"
with open(meta_path, "w", encoding="utf-8") as fo:
    json.dump({"K": K, "M": M, "SEED": SEED, "docs": chosen_docs}, fo, ensure_ascii=False, indent=2)
print("wrote:", meta_path)

def export_method_subset(method: str, docs, T: int, out_path: Path, seed_offset=0):
    """
    Writes exactly M*K rows for this method.
    For QA we pass different seed_offset per repeat to resample k outputs/doc.
    """
    rows_written = 0
    with open(out_path, "w", encoding="utf-8") as fo:
        for doc in docs:
            texts = G[method][doc]
            # deterministic selection
            seed_key = (SEED + seed_offset, method, doc)
            picks = pick_k(texts, K, seed_key=hash(seed_key))

            for j, t in enumerate(picks):
                if DO_CLEAN:
                    t = clean_basic(t)
                t = truncate_random_window(t, T, seed_key=(SEED, method, doc, f))

                rec = {
                    "method": method,
                    "doc_name": doc,
                    "sample_id": j,
                    "text": t,
                    "T_words": T,
                    "K": K,
                    "seed": SEED,
                    "seed_offset": seed_offset,
                }
                fo.write(json.dumps(rec, ensure_ascii=False) + "\n")
                rows_written += 1
    return rows_written

# ---- Export for each T ----
for T in T_LIST:
    # fixed subsets for AR and paraphrase
    ar_path = OUTDIR / f"active_reading_eval_k{K}_M{M}_T{T}_seed{SEED}.jsonl"
    pa_path = OUTDIR / f"paraphrase_eval_k{K}_M{M}_T{T}_seed{SEED}.jsonl"

    n_ar = export_method_subset("active_reading", chosen_docs, T, ar_path, seed_offset=0)
    n_pa = export_method_subset("paraphrase", chosen_docs, T, pa_path, seed_offset=0)

    print("wrote:", ar_path, "rows:", n_ar)
    print("wrote:", pa_path, "rows:", n_pa)

    # QA: multiple repeats to quantify selection variance
    for r in range(QA_REPEATS):
        qa_path = OUTDIR / f"qa_eval_k{K}_M{M}_T{T}_seed{SEED}_rep{r:02d}.jsonl"
        n_qa = export_method_subset("qa", chosen_docs, T, qa_path, seed_offset=r+1)  # offset changes sampling
        if r == 0:
            print("wrote:", qa_path, "rows:", n_qa)

print("\nDone. Eval sets are in:", OUTDIR)


active_reading: docs=735
qa: docs=855
paraphrase: docs=856
docs in all 3: 614
eligible (>=K=3 in all 3): 78
chosen_docs: 78
wrote: Finetune_Datasets/simplewiki/eval_sets/meta_k3_M78_seed42.json
wrote: Finetune_Datasets/simplewiki/eval_sets/active_reading_eval_k3_M78_T128_seed42.jsonl rows: 234
wrote: Finetune_Datasets/simplewiki/eval_sets/paraphrase_eval_k3_M78_T128_seed42.jsonl rows: 234
wrote: Finetune_Datasets/simplewiki/eval_sets/qa_eval_k3_M78_T128_seed42_rep00.jsonl rows: 234
wrote: Finetune_Datasets/simplewiki/eval_sets/active_reading_eval_k3_M78_T256_seed42.jsonl rows: 234
wrote: Finetune_Datasets/simplewiki/eval_sets/paraphrase_eval_k3_M78_T256_seed42.jsonl rows: 234
wrote: Finetune_Datasets/simplewiki/eval_sets/qa_eval_k3_M78_T256_seed42_rep00.jsonl rows: 234

Done. Eval sets are in: Finetune_Datasets/simplewiki/eval_sets


# Self-BLEU

Για κάθε doc με k outputs:

υπολογίζουμε BLEU κάθε output ως hypothesis έναντι των υπολοίπων ως references

κάνουμε average (self-BLEU) ανά doc

μετά average σε όλα τα docs

μικρότερο self-BLEU ⇒ μεγαλύτερη diversity (λιγότερη ομοιότητα μεταξύ outputs).

In [83]:
import json, re, math, glob
from pathlib import Path
from collections import defaultdict, Counter
import numpy as np
import pandas as pd

EVAL_DIR = Path("Finetune_Datasets/simplewiki/eval_sets")

def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rec = json.loads(line)
            rows.append(rec)
    return rows

def tokenize(text: str):
    # simple, deterministic tokenizer (consistent across methods)
    text = text.lower()
    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text.split()

def ngrams(tokens, n):
    if len(tokens) < n:
        return []
    return list(zip(*[tokens[i:] for i in range(n)]))

# Try to use NLTK BLEU if available; otherwise fallback to 4-gram Jaccard
USE_NLTK = False
try:
    from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
    USE_NLTK = True
    _SMOOTH = SmoothingFunction().method1
except Exception:
    USE_NLTK = False
    _SMOOTH = None

def self_bleu4(texts):
    # texts: list[str], length k>=2
    toks = [tokenize(t) for t in texts]
    if len(toks) < 2:
        return np.nan

    if USE_NLTK:
        scores = []
        for i in range(len(toks)):
            hyp = toks[i]
            refs = [toks[j] for j in range(len(toks)) if j != i]
            scores.append(sentence_bleu(
                refs, hyp,
                weights=(0.25,0.25,0.25,0.25),
                smoothing_function=_SMOOTH
            ))
        return float(np.mean(scores))

    # fallback proxy (higher = more similar => lower diversity)
    grams = [set(ngrams(t, 4)) for t in toks]
    sims = []
    for i in range(len(grams)):
        for j in range(i+1, len(grams)):
            if not grams[i] and not grams[j]:
                continue
            inter = len(grams[i] & grams[j])
            union = len(grams[i] | grams[j])
            sims.append(inter / union if union else 0.0)
    return float(np.mean(sims)) if sims else np.nan



def compute_metrics_for_file(path):
    rows = load_jsonl(path)
    df = pd.DataFrame(rows)
    method = df["method"].iloc[0]
    T = int(df["T_words"].iloc[0])
    rep = int(df["seed_offset"].iloc[0])  # 0 for AR/PAR, >0 for QA repeats
    # group by doc
    per_doc = []
    for doc, g in df.groupby("doc_name"):
        texts = g.sort_values("sample_id")["text"].tolist()
        per_doc.append({
            "doc_name": doc,
            "self_bleu4": self_bleu4(texts),
        })
    per_doc_df = pd.DataFrame(per_doc)
    return method, T, rep, per_doc_df


In [82]:
files = sorted(glob.glob(str(EVAL_DIR / "*_eval_k3_M78_T*_seed42*.jsonl")))

records = []

for fp in files:
    method, T, rep, per_doc_df = compute_metrics_for_file(fp)

    # average across docs
    m_bleu = float(per_doc_df["self_bleu4"].mean())


    records.append({
        "method": method,
        "T_words": T,
        "rep": rep,
        "self_bleu4_mean": m_bleu,
        "docs_used": int(len(per_doc_df)),
        "bleu_impl": "nltk_bleu4" if USE_NLTK else "fallback_4gram_jaccard_proxy",
    })

res = pd.DataFrame(records)

# Split fixed (AR/PAR) vs QA repeats
fixed = res[(res["method"].isin(["active_reading","paraphrase"]))].copy()

qa = res[res["method"].eq("qa")].copy()
# For QA: aggregate over repeats -> mean ± 95% CI
def mean_ci(x):
    x = np.array(x, dtype=float)
    mu = float(x.mean())
    sd = float(x.std(ddof=1)) if len(x) > 1 else 0.0
    se = sd / math.sqrt(len(x)) if len(x) > 0 else np.nan
    ci = 1.96 * se
    return mu, ci

qa_rows = []
for T in sorted(qa["T_words"].unique()):
    sub = qa[qa["T_words"].eq(T)]
    mu_bleu, ci_bleu = mean_ci(sub["self_bleu4_mean"])
    qa_rows.append({
        "method": "qa",
        "T_words": T,
        "self_bleu4_mean": mu_bleu,
        "self_bleu4_95ci": ci_bleu,
        "repeats": int(len(sub)),
        "docs_used": int(sub["docs_used"].iloc[0]) if len(sub) else 0,
        "bleu_impl": sub["bleu_impl"].iloc[0] if len(sub) else "",
    })

qa_summary = pd.DataFrame(qa_rows)

# Combine into final table
fixed_table = fixed.groupby(["method","T_words"], as_index=False).agg(
    self_bleu4_mean=("self_bleu4_mean","mean"),
    docs_used=("docs_used","mean"),
    bleu_impl=("bleu_impl","first"),
)

final = pd.concat([
    fixed_table.assign(self_bleu4_95ci=np.nan, repeats=np.nan),
    qa_summary
], ignore_index=True)

final = final.sort_values(["T_words","method"]).reset_index(drop=True)

print("=== FINAL (intra-doc) ===")
display(final)


=== FINAL (intra-doc) ===


,method,T_words,self_bleu4_mean,docs_used,bleu_impl,self_bleu4_95ci,repeats
0,active_reading,128,0.020546,78.0,nltk_bleu4,NaN,NaN
1,paraphrase,128,0.015938,78.0,nltk_bleu4,NaN,NaN
2,qa,128,0.079441,78.0,nltk_bleu4,0.003774,50.0
3,active_reading,256,0.032062,78.0,nltk_bleu4,NaN,NaN
4,paraphrase,256,0.026528,78.0,nltk_bleu4,NaN,NaN
5,qa,256,0.079451,78.0,nltk_bleu4,0.003771,50.0


# Σύγκριση όλων των μεθόδων
Σε πλήρως αντιστοιχισμένο υποσύνολο (k=3,M=78, κοινά doc_name σε active reading, paraphrase και QA), το Self-BLEU-4 υποδεικνύει ότι η μέθοδος paraphrase παράγει τα πιο διαφοροποιημένα outputs για το ίδιο έγγραφο, καθώς παρουσιάζει σταθερά τις χαμηλότερες τιμές. Το active reading εμφανίζει ενδιάμεση συμπεριφορά και βελτιώνεται αισθητά με την αύξηση του word budget (0.0723 σε 0.0593), γεγονός που είναι συνεπές με το ότι μεγαλύτερο διαθέσιμο μήκος επιτρέπει μεγαλύτερη απόκλιση από κοινά δομικά πρότυπα. Αντίθετα, το QA παρουσιάζει τις υψηλότερες τιμές (0.0795 και στα δύο budgets), δηλαδή τη μικρότερη λεξική ποικιλία κατά το συγκεκριμένο metric. Επιπλέον, μέσω repeated subsampling (50 επαναλήψεις) εκτιμάται η αβεβαιότητα της επιλογής των 3 outputs στο QA (95% CI ±0.00377), δείχνοντας ότι το αποτέλεσμα είναι σταθερό ως προς τη δειγματοληψία.

Η δεύτερη σύγκριση (active reading vs paraphrase) πραγματοποιείται συμπληρωματικά για δύο λόγους.

Πρώτον, το QA συνιστά ποιοτικά διαφορετικό τύπο generation, με αποτέλεσμα η σύγκριση όλων των μεθόδων μαζί να επηρεάζεται από διαφορές task και μήκους που δεν είναι πλήρως συγκρίσιμες, ακόμη και υπό fixed-budget περικοπή. Δεύτερον, αφαιρώντας το QA διευρύνεται το εφικτό αντιστοιχισμένο σύνολο docs αυξάνοντας τη στατιστική ισχύ και επιτρέποντας καθαρότερη απομόνωση της επίδρασης του word budget σε δύο μεθόδους με παρόμοιο τύπο εξόδου (μακρύτερα κείμενα).

In [84]:
import json, re, random
from pathlib import Path
from collections import defaultdict

BASE = Path("Finetune_Datasets/simplewiki")

AR_PATH = BASE / "active_reading_dataset.jsonl"
PA_PATH = BASE / "paraphrase_dataset.jsonl"

OUTDIR2 = BASE / "eval_sets_ar_vs_para"
OUTDIR2.mkdir(parents=True, exist_ok=True)

# ====== PARAMS ======
K = 3
T_LIST = [128, 256, 512]   # add 512 to show trend
SEED = 42
DO_CLEAN = True
# ====================

def canon_doc_name(doc: str, method: str) -> str:
    doc = "" if doc is None else str(doc)
    doc = doc.strip()
    doc = re.sub(r"\s+", " ", doc)
    return doc

def get_text(rec, method):
    if method == "paraphrase":
        return (rec.get("text") or "").strip()
    return (rec.get("active_reading") or "").strip()

def clean_basic(text: str) -> str:
    text = re.sub(r"^#+\s*", "", text, flags=re.MULTILINE)
    text = re.sub(r"^---+\s*$", " ", text, flags=re.MULTILINE)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def truncate_random_window(text: str, T: int, seed_key):
    w = text.split()
    if len(w) <= T:
        return text
    rng = random.Random(hash(seed_key))
    start = rng.randrange(0, len(w) - T + 1)
    return " ".join(w[start:start+T])

def load_grouped(path: Path, method: str):
    g = defaultdict(list)
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rec = json.loads(line)
            doc = canon_doc_name(rec.get("doc_name", ""), method)
            text = get_text(rec, method)
            if text:
                g[doc].append(text)
    return g

def pick_k(texts, k, seed_key):
    rng = random.Random(seed_key)
    idx = list(range(len(texts)))
    rng.shuffle(idx)
    return [texts[i] for i in idx[:k]]

G_ar = load_grouped(AR_PATH, "active_reading")
G_pa = load_grouped(PA_PATH, "paraphrase")

docs_common = set(G_ar.keys()) & set(G_pa.keys())
eligible = sorted([d for d in docs_common if len(G_ar[d]) >= K and len(G_pa[d]) >= K])

print("AR docs:", len(G_ar), "PARA docs:", len(G_pa))
print("common docs:", len(docs_common))
print(f"eligible (>=K={K} in both):", len(eligible))

if len(eligible) == 0:
    raise ValueError("No eligible docs. Reduce K or check datasets.")

# take maximum available (or cap if you want)
M = len(eligible)  # use all eligible for strongest power
print("Using M =", M)

rng = random.Random(SEED)
chosen = eligible.copy()
rng.shuffle(chosen)
chosen = sorted(chosen[:M])

meta = OUTDIR2 / f"meta_k{K}_M{M}_seed{SEED}.json"
with open(meta, "w", encoding="utf-8") as fo:
    json.dump({"K": K, "M": M, "SEED": SEED, "docs": chosen}, fo, ensure_ascii=False, indent=2)
print("wrote:", meta)

def export(method, G, docs, T, out_path):
    rows = 0
    with open(out_path, "w", encoding="utf-8") as fo:
        for doc in docs:
            picks = pick_k(G[doc], K, seed_key=hash((SEED, method, doc)))
            for sid, t in enumerate(picks):
                if DO_CLEAN:
                    t = clean_basic(t)
                t = truncate_random_window(t, T, seed_key=(SEED, method, doc, sid))

                fo.write(json.dumps({
                    "method": method,
                    "doc_name": doc,
                    "sample_id": sid,
                    "text": t,
                    "T_words": T,
                    "K": K,
                    "seed": SEED,
                }, ensure_ascii=False) + "\n")
                rows += 1
    return rows

for T in T_LIST:
    ar_out = OUTDIR2 / f"active_reading_eval_k{K}_M{M}_T{T}_seed{SEED}.jsonl"
    pa_out = OUTDIR2 / f"paraphrase_eval_k{K}_M{M}_T{T}_seed{SEED}.jsonl"
    print("wrote:", ar_out, "rows:", export("active_reading", G_ar, chosen, T, ar_out))
    print("wrote:", pa_out, "rows:", export("paraphrase", G_pa, chosen, T, pa_out))

print("Done. Eval sets in:", OUTDIR2)


AR docs: 735 PARA docs: 856
common docs: 733
eligible (>=K=3 in both): 85
Using M = 85
wrote: Finetune_Datasets/simplewiki/eval_sets_ar_vs_para/meta_k3_M85_seed42.json
wrote: Finetune_Datasets/simplewiki/eval_sets_ar_vs_para/active_reading_eval_k3_M85_T128_seed42.jsonl rows: 255
wrote: Finetune_Datasets/simplewiki/eval_sets_ar_vs_para/paraphrase_eval_k3_M85_T128_seed42.jsonl rows: 255
wrote: Finetune_Datasets/simplewiki/eval_sets_ar_vs_para/active_reading_eval_k3_M85_T256_seed42.jsonl rows: 255
wrote: Finetune_Datasets/simplewiki/eval_sets_ar_vs_para/paraphrase_eval_k3_M85_T256_seed42.jsonl rows: 255
wrote: Finetune_Datasets/simplewiki/eval_sets_ar_vs_para/active_reading_eval_k3_M85_T512_seed42.jsonl rows: 255
wrote: Finetune_Datasets/simplewiki/eval_sets_ar_vs_para/paraphrase_eval_k3_M85_T512_seed42.jsonl rows: 255
Done. Eval sets in: Finetune_Datasets/simplewiki/eval_sets_ar_vs_para


In [85]:
def compute_metrics_for_file(path):
    rows = load_jsonl(path)
    df = pd.DataFrame(rows)
    method = df["method"].iloc[0]
    T = int(df["T_words"].iloc[0])

    # NEW: seed_offset is optional (exists in QA repeat files)
    rep = int(df["seed_offset"].iloc[0]) if "seed_offset" in df.columns else 0

    per_doc = []
    for doc, g in df.groupby("doc_name"):
        texts = g.sort_values("sample_id")["text"].tolist()
        per_doc.append({
            "doc_name": doc,
            "self_bleu4": self_bleu4(texts),
            # keep ent2 if you want; otherwise omit
            # "ent2": ent2(texts, normalized=True),
        })
    per_doc_df = pd.DataFrame(per_doc)
    return method, T, rep, per_doc_df


In [87]:
import glob
import pandas as pd
import numpy as np
from pathlib import Path

EVAL2 = Path("Finetune_Datasets/simplewiki/eval_sets_ar_vs_para")

files = sorted(glob.glob(str(EVAL2 / "*_eval_k3_M*_T*_seed42.jsonl")))

rows = []
for fp in files:
    method, T, rep, per_doc_df = compute_metrics_for_file(fp)  # from your earlier cell
    rows.append({
        "method": method,
        "T_words": int(T),
        "self_bleu4_mean": float(per_doc_df["self_bleu4"].mean()),
        "docs_used": int(len(per_doc_df)),
        "bleu_impl": "nltk_bleu4" if USE_NLTK else "fallback_proxy",
    })

table = pd.DataFrame(rows).sort_values(["T_words","method"]).reset_index(drop=True)
display(table)





,method,T_words,self_bleu4_mean,docs_used,bleu_impl
0,active_reading,128,0.020200,85,nltk_bleu4
1,paraphrase,128,0.020288,85,nltk_bleu4
2,active_reading,256,0.026083,85,nltk_bleu4
3,paraphrase,256,0.029273,85,nltk_bleu4
4,active_reading,512,0.041339,85,nltk_bleu4
5,paraphrase,512,0.040148,85,nltk_bleu4


In [73]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

def df_to_png_table(df: pd.DataFrame, out_png: str, title: str = None, dpi: int = 220):
    df = df.copy()

    # prettier column names (optional)
    df.columns = [c.replace("_", " ") for c in df.columns]

    # stringify NaN as "—"
    df = df.replace({np.nan: "—"})

    # round numeric columns for presentation
    for c in df.columns:
        if df[c].dtype != object:
            df[c] = df[c].map(lambda x: f"{x:.6f}" if isinstance(x, (float, np.floating)) else x)

    nrows, ncols = df.shape

    # figure size heuristic: scale with table dimensions
    fig_w = max(8, 1.2 * ncols)
    fig_h = max(2.2, 0.5 * nrows + (0.8 if title else 0.2))

    fig, ax = plt.subplots(figsize=(fig_w, fig_h))
    ax.axis("off")

    if title:
        ax.set_title(title, fontsize=14, pad=12)

    # Create table
    table = ax.table(
        cellText=df.values,
        colLabels=df.columns,
        cellLoc="center",
        colLoc="center",
        loc="center",
    )

    # styling
    table.auto_set_font_size(False)
    table.set_fontsize(11)
    table.scale(1, 1.5)

    # bold header row
    for (row, col), cell in table.get_celld().items():
        if row == 0:
            cell.set_text_props(weight="bold")
        cell.set_linewidth(0.5)

    out_path = Path(out_png)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    plt.tight_layout()
    plt.savefig(out_path, dpi=dpi, bbox_inches="tight")
    plt.close(fig)

    return str(out_path)



png1 = df_to_png_table(final, "exports/selfbleu_main_matched.png", title="Self-BLEU-4 — matched across all methods (k=3, M=78)")


png2 = df_to_png_table(table, "exports/selfbleu_ar_vs_para.png",
                      title="Self-BLEU-4 (intra-doc) — Active Reading vs Paraphrase (k=3)")


print("Uncomment one of the examples and run to export PNG.")


Uncomment one of the examples and run to export PNG.
